In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)
with open("input.txt", "r") as f:
    text = f.read()[:50000]  # use first 50k chars to keep training fast
    chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(chars)
SEQ_LEN = 40
STEP = 3
X_data, y_data = [], []
for i in range(0, len(text) - SEQ_LEN, STEP):
    X_data.append([char2idx[c] for c in text[i:i+SEQ_LEN]])
    y_data.append(char2idx[text[i+SEQ_LEN]])
X = np.zeros((len(X_data), SEQ_LEN, vocab_size), dtype=np.float32)
y = np.zeros((len(y_data), vocab_size), dtype=np.float32)
for i, seq in enumerate(X_data):
    for t, idx in enumerate(seq):
        X[i, t, idx] = 1.0
    y[i, y_data[i]] = 1.0
# 4. Build character-level RNN
model = Sequential([
    LSTM(128, input_shape=(SEQ_LEN, vocab_size)),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()
model.fit(X, y, batch_size=128, epochs=10, verbose=1)
def generate_text(seed_text, num_chars=200, temperature=0.5):
    seed_text = seed_text[-SEQ_LEN:].ljust(SEQ_LEN)
    generated = seed_text
    for _ in range(num_chars):
        x_pred = np.zeros((1, SEQ_LEN, vocab_size))
        for t, c in enumerate(seed_text):
            if c in char2idx:
                x_pred[0, t, char2idx[c]] = 1.0
        preds = model.predict(x_pred, verbose=0)[0]
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))
        next_idx = np.random.choice(len(preds), p=preds)
        next_char = idx2char[next_idx]
        generated += next_char
        seed_text = seed_text[1:] + next_char
    return generated
seed = "First Citizen:"
print("\nGENERATED TEXT")
print(generate_text(seed, num_chars=300, temperature=0.5))


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 128)            │        96,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 59)             │         7,611 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,867 (405.73 KB)

 Trainable params: 103,867 (405.73 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 16s 106ms/step - loss: 3.3558
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 107ms/step - loss: 3.1245
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 108ms/step - loss: 2.7934
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 105ms/step - loss: 2.5778
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 105ms/step - loss: 2.4665
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 21s 105ms/step - loss: 2.3825
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 105ms/step - loss: 2.3218
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 105ms/step - loss: 2.2736
Epoch 9/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - loss: 2.2266
Epoch 10/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 14s 107ms/step - loss: 2.1972

GENERATED TEXT
First Citizen:                          iovine wor ond the he gore the hat he the the he heall the pour the memy fou dithen thet ond wind wous were de hor in mithor thave ram and and and ore in hericher on the fome ou gouind and and wo hour the coprour the hithing and the mand and t